# Lekce 18: Zabezpečení AI agentů pomocí kryptografických potvrzení

## Praktický sešit

Tento sešit vás provede čtyřmi cvičeními:

1. **Podepište své první potvrzení** pro volání nástroje agenta a ověřte jej.
2. **Poškoďte potvrzení** a sledujte selhání ověření.
3. **Sestavte řetězec ze tří potvrzení** a potvrďte integritu řetězce.
4. **Zabalte volání nástroje Microsoft Agent Framework** tak, aby každá akce vydávala potvrzení.

Všechny kryptografické primitivy jsou importovány z dobře udržovaných knihoven (`pynacl` pro Ed25519, `jcs` pro RFC 8785 canonical JSON, `hashlib` z Python standardní knihovny pro SHA-256). Logika potvrzení je samotný čistý Python, který můžete číst a upravovat.

Spouštějte buňky po pořadí. Každá sekce je krátká a samostatná.


## Nastavení

Nainstalujte dvě závislosti. Obě mají povolující licence (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Pomocné nástroje

Tito dva pomocníci zpracovávají base64url kódování (bez doplnění) a SHA-256 hashování libovolných objektů. Umožňují, aby zbytek notebooku zůstal zaměřen na logiku účtenky samotné.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Podepište svůj první doklad

Představte si, že náš agent pro **Contoso Travel** právě vyhledal lety ze Sydney do Los Angeles pro zákazníka. Chceme tento nástrojový požadavek zaznamenat jako podepsaný doklad, aby jej budoucí auditor mohl ověřit bez nutnosti nám důvěřovat.

### Krok 1.1: Vygenerujte podepisovací klíč

V produkci by podepisovací klíč agenta žil v hardwarovém bezpečnostním modulu (HSM), Azure Key Vault nebo podobném chráněném úložišti. Pro tuto lekci generujeme čerstvý klíč v paměti.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Krok 1.2: Vytvoření obsahu účtenky

Obsah obsahuje vše, co chceme, aby účtenka potvrdila: kdo jednal, jaký nástroj, s jakými argumenty, co se vrátilo, podle jaké politiky a kdy. Argumenty a výsledek hashujeme místo toho, abychom je uváděli přímo, aby účtenka neunikala citlivý obsah.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Krok 1.3: Podepište a sestavte potvrzení

Tři kroky:

1. Kanonizujte payload pomocí JCS tak, aby dvě implementace produkující stejné logické potvrzení vytvářely bytově totožné bajty.
2. Zahashujte kanonické bajty pomocí SHA-256.
3. Podepište hash pomocí privátního klíče Ed25519.

Poté je podpis připojen k původnímu payloadu, aby vzniklo konečné potvrzení.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Krok 1.4: Ověření účtenky

Ověření proces obrací. Odstraníme podpis, znovu spočítáme kanonický hash a ověříme podpis vůči veřejnému klíči v účtence.

Auditora provádějícího toto ověření nepotřebujeme nic kromě samotné účtenky. Žádná služba k volání, žádný adresář klíčů k dotazu, žádná důvěra není vyžadována.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Měli byste vidět `Receipt is valid: True`. Agent vytvořil svůj první kryptograficky podepsaný auditní záznam.


## Sekce 2: Zmanipulujte účtenku

Celý smysl účtenek je, že jsou odolné proti manipulaci. Dokážeme to.

Upravíme přesně jeden znak na účtence a sledujeme, jak ověření selže.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Co se právě stalo?

Když jsme změnili `policy_id`, změnily se kanonické bajty. SHA-256 hash těchto bajtů se změnil. Podpis (který byl nad původním hashem) už neodpovídá novému hashi. Ověření správně vrací `False`.

Není možné upravit žádné pole účtenky a přitom ji nechat ověřit, pokud nemá útočník soukromý klíč. Dokud je soukromý klíč uložen v klíčové schránce a veřejný klíč je zveřejněn, nelze manipulaci skrýt.

Vyzkoušejte sami: změňte `tool_name` nebo `agent_id` nebo `timestamp` v buňce výše a spusťte znovu. Každá změna vytvoří neplatnou účtenku.


## Sekce 3: Řetězení účtenek dohromady

Jedna účtenka chrání jednu akci. Většina agentů provádí mnoho akcí. Aby byl celý sled odolný proti manipulaci, propojujeme každou účtenku s předchozí tím, že do dat nové účtenky zahrneme hash předchozí účtenky.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Pokud někdo účtenku odstraní nebo změní pořadí, řetězec se v tomto bodě přeruší. Ověření jakékoliv pozdější účtenky selže, protože její `previous_receipt_hash` již neodpovídá skutečnému hashi jejího předchůdce.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Nyní přerušte řetězec tím, že pozměníte prostřední účtenku a znovu ověříte. Pozměněná účtenka neprojde kontrolou podpisu, A následující účtenka neprojde kontrolou řetězového odkazu (protože její `previous_receipt_hash` již neodpovídá hash hodnotě pozměněné prostřední účtenky).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Příjemka 0 stále ověřuje (nebyla upravena a nemá předchůdce, na kterém by závisela). Příjemka 1 neprochází kontrolou podpisu, protože jsme změnili `tool_args_hash`. Příjemka 2 neprochází kontrolou řetězového odkazu, protože její `previous_receipt_hash` byl vypočítán vůči původní (nyní upravené) příjemce 1.

I kdyby útočník znovu podepsal upravenou příjemku 1 (což nemůže udělat bez soukromého klíče), nesoulad řetězového odkazu v příjemce 2 by stále odhalil manipulaci. Aby změnu skryl, musel by útočník znovu podepsat každou příjemku od místa úpravy dál, což vyžaduje vlastníctví soukromého klíče.


## Sekce 4: Zabalte volání nástroje agenta s podepisováním potvrzení

V reálném nasazení nechcete, aby si každý autor agenta pamatoval volat `make_receipt`. Chcete, aby podepisování potvrzení bylo automatické při každém volání nástroje.

Zde je nejjednodušší vzor: obálková třída, která vezme libovolnou volatelnou funkci nástroje a vrátí její verzi emitující potvrzení. Toto se přizpůsobí jakémukoli agentovému frameworku, včetně Microsoft Agent Framework (`agent_framework.azure`).

Pokud nemáte nastavený projekt Azure AI Foundry, níže uvedený místní mock stále demonstruje tento vzor.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integrace s Microsoft Agent Framework

Výše uvedený obal `ReceiptedTool` je nezávislý na frameworku. Pro použití uvnitř agenta postaveného na Microsoft Agent Framework zaregistrujte zabalenou funkci jako nástroj. Náčrt (nahradíte mock skutečnou registrací nástroje Azure AI Foundry):

```python
# Pseudokód zobrazující tvar integrace.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Jste agentem Contoso Travel ...",
#     tools=[receipted_lookup],   # zabalený nástroj, ne surová funkce
# )
# response = agent.run("Najděte lety ze Sydney do Los Angeles v červnu.")
#
# # Po spuštění má každé volání nástroje, které agent provedl, podepsaný doklad:
# audit_chain = receipted_lookup.receipts
```

Agent framework nemusí o účtenkách nic vědět. Podepisování účtenek je zabaleno kolem nástroje, není integrováno do frameworku. Takto přidáte původ dat k existujícímu kódu agenta, aniž byste přepisovali agenta.


## Rekapitulace a rozšiřující úkol

Máte:

- Vygenerovanou Ed25519 pár klíčů.
- Vytvořený a podepsaný potvrzení o volání nástroje agenta.
- Offline ověřené potvrzení pouze pomocí veřejného klíče.
- Zfalšované potvrzení a pozorované selhání ověření.
- Vytvořenou hashově propojenou sekvenci tří potvrzení.
- Zfalšovanou prostřední část řetězce a pozorované selhání jak podpisu, tak propojení řetězce.
- Zabalenou funkci nástroje s automatickým podepisováním potvrzení.

**Rozšiřující úkol.** Rozšiřte schéma potvrzení o pole `request_id` (UUID pro distribuované trasování). Aktualizujte `make_receipt`, aby jej zahrnovala, a ověřte, že potvrzení nadále ověřují od začátku do konce. Pak pole po podepsání změňte a potvrďte, že ověření selže. To vás nutí si uvědomit, jak každý bajt kanonického kódování přispívá k podpisu.

**Důležitá hranice.** Potvrzení dokazují tři věci a pouze tři věci: přiřazení (tento klíč podepsal tento obsah), integritu (obsah se od podpisu nezměnil) a pořadí (toto potvrzení přišlo po tom potvrzení). NEdokazují, že akce agenta byla správná, že politika pojmenovaná v `policy_id` byla skutečně vyhodnocena, ani že agent dodržel všechna pravidla. Potvrzení jsou základ. Správa je systém, který na tom stavíte.

Přečtěte si znovu README lekce s touto hranicí na paměti. Nejčastější chybou týmů s potvrzeními je předpoklad, že "máme potvrzení" znamená "jsme spravováni". Není to tak. Potvrzení umožňují auditovatelnost chování agenta. Nezaručují jeho správnost.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
